# SOCat Interactive Notebook

Welcome — this in-browser notebook lets you explore the SOCat catalog directly with Python (Pyodide). The catalog is fetched live from the SOCat website on every run, so you always get the latest version.

**Quick demo:** sky-projected obliquity (λ) vs. host effective temperature.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Use STIX (Times-like) font for all plots — matplotlib-bundled, no download needed
plt.rcParams['font.family']      = 'STIXGeneral'
plt.rcParams['mathtext.fontset'] = 'stix'

# Fetch the catalog directly from the SOCat website (same origin, no CORS issue).
# `comment='#'` skips the version / column-definition header.
SOCAT_CSV_URL = 'https://www.stellarobliquity.com/app/static/obliquitywwb.csv'
df = pd.read_csv(SOCAT_CSV_URL, comment='#')
print(f'Loaded {len(df)} obliquity measurements across {df["hostname"].nunique()} host stars')
df.head()

## λ vs T$_{\rm eff}$

Each point is one obliquity measurement; horizontal/vertical bars show 1σ uncertainties. λ is normalized to [0°, 180°] below (negative → +360, then folded around 180).

In [ ]:
def normalize_lambda(lam, err1, err2):
    """Fold λ into [0, 180]. NEA-style err1=+upper, err2=-lower (negative)."""
    lam_out, lo_out, hi_out = [], [], []
    for v, e1, e2 in zip(lam, err1, err2):
        if pd.isna(v):
            lam_out.append(np.nan); lo_out.append(np.nan); hi_out.append(np.nan); continue
        x = v + 360 if v < 0 else v
        if 0 <= x < 180:
            lam_out.append(x);          lo_out.append(abs(e2)); hi_out.append(abs(e1))
        else:
            lam_out.append(360 - x);    lo_out.append(abs(e1)); hi_out.append(abs(e2))
    return np.array(lam_out), np.array(lo_out), np.array(hi_out)

good = df.dropna(subset=['st_teff', 'pl_projobliq']).copy()
good['lam_n'], good['lam_lo'], good['lam_hi'] = normalize_lambda(
    good['pl_projobliq'].values,
    good['pl_projobliqerr1'].values,
    good['pl_projobliqerr2'].values,
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(
    good['st_teff'],
    good['lam_n'],
    xerr=[good['st_tefferr2'].abs(), good['st_tefferr1'].abs()],
    yerr=[good['lam_lo'],            good['lam_hi']],
    fmt='o', ms=4, lw=0.6, alpha=0.7, color='C0',
)
ax.axvline(6250, color='C3', ls='--', lw=1, alpha=0.6, label='Kraft break (~6250 K)')
ax.set_xlabel(r'Host T$_{\rm eff}$ (K)')
ax.set_ylabel(r'$|\lambda|$ (deg, normalized to [0, 180])')
ax.set_title('SOCat: sky-projected obliquity vs. host effective temperature')
ax.set_ylim(-5, 185)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f'Plotted {len(good)} of {len(df)} rows (others missing Teff or λ).')

## Try your own analysis

All catalog columns are listed below. Add a cell and explore freely — the notebook autosaves to your browser storage.

In [ ]:
list(df.columns)